# Notebook 08 — Model Comparison: GraphSAGE vs GCN

This notebook provides a side-by-side comparison of the heterogeneous GraphSAGE and GCN models trained in notebooks 06 and 07. We evaluate them on:
1.  **Seen-User Performance**: Transductive setting (test edges from users original in the training graph).
2.  **Hidden-User Performance**: Inductive setting (cold-start users held out during training).
3.  **Inductive Gap**: The drop in performance when moving from seen to hidden users.

In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
PROCESSED_DIR = Path("../data/processed/")

sage_path = PROCESSED_DIR / "graphsage_link_pred.pt"
gcn_path = PROCESSED_DIR / "gcn_link_pred.pt"

if not sage_path.exists() or not gcn_path.exists():
    print("Warning: Checkpoints missing. Ensure Notebooks 06 and 07 have been run.")
else:
    sage = torch.load(sage_path, weights_only=False)
    gcn = torch.load(gcn_path, weights_only=False)
    print("Checkpoints loaded successfully.")

## 1. Training History (GCN Baseline)
GraphSAGE was trained using neighborhood sampling (LinkNeighborLoader) for 5 epochs, while GCN was trained on the full-graph for up to 25 epochs with early stopping.

In [ ]:
if 'seen_user_training' in gcn and gcn['seen_user_training']['ok']:
    history_df = pd.DataFrame(gcn['seen_user_training']['value'])
    
    fig, ax1 = plt.subplots()

    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss', color='tab:red')
    ax1.plot(history_df['epoch'], history_df['train_loss'], color='tab:red', label='Train Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Val AUROC', color='tab:blue')
    ax2.plot(history_df['epoch'], history_df['val_auroc'], color='tab:blue', label='Val AUROC')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    plt.title('GCN Full-Graph Training History')
    fig.tight_layout()
    plt.show()
else:
    print("GCN training history not available.")

## 2. Final Performance Comparison

We compare AUROC and Average Precision (AP) for both models across Seen and Hidden users.

In [ ]:
rows = []

# GraphSAGE
rows.append({"Model": "GraphSAGE", "Split": "Seen Users", "Metric": "AUROC", "Value": sage['test_seen_users_edge_split']['auroc']})
rows.append({"Model": "GraphSAGE", "Split": "Seen Users", "Metric": "AP", "Value": sage['test_seen_users_edge_split']['ap']})
rows.append({"Model": "GraphSAGE", "Split": "Hidden Users", "Metric": "AUROC", "Value": sage['inductive_hidden_user_test']['auroc']})
rows.append({"Model": "GraphSAGE", "Split": "Hidden Users", "Metric": "AP", "Value": sage['inductive_hidden_user_test']['ap']})

# GCN
if gcn['seen_user_test']['ok']:
    rows.append({"Model": "GCN", "Split": "Seen Users", "Metric": "AUROC", "Value": gcn['seen_user_test']['value']['auroc']})
    rows.append({"Model": "GCN", "Split": "Seen Users", "Metric": "AP", "Value": gcn['seen_user_test']['value']['ap']})
if gcn['inductive_hidden_user_test']['ok']:
    rows.append({"Model": "GCN", "Split": "Hidden Users", "Metric": "AUROC", "Value": gcn['inductive_hidden_user_test']['value']['auroc']})
    rows.append({"Model": "GCN", "Split": "Hidden Users", "Metric": "AP", "Value": gcn['inductive_hidden_user_test']['value']['ap']})

metrics_df = pd.DataFrame(rows)

plt.figure(figsize=(12, 6))
sns.barplot(data=metrics_df, x="Metric", y="Value", hue="Model", style="Split", palette="viridis")
plt.title("GraphSAGE vs GCN: Metric Comparison")
plt.ylim(0.7, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. The Inductive Gap

How much does performance drop on users the model has never seen?

In [ ]:
gap_data = {
    'Model': ['GraphSAGE', 'GCN'],
    'Seen AUROC': [
        sage['test_seen_users_edge_split']['auroc'],
        gcn['seen_user_test']['value']['auroc']
    ],
    'Hidden AUROC': [
        sage['inductive_hidden_user_test']['auroc'],
        gcn['inductive_hidden_user_test']['value']['auroc']
    ]
}
gap_df = pd.DataFrame(gap_data)
gap_df['Gap'] = gap_df['Seen AUROC'] - gap_df['Hidden AUROC']

plt.figure(figsize=(8, 5))
sns.barplot(data=gap_df, x='Model', y='Gap', palette="magma")
plt.title("Inductive AUROC Gap (Lower is Better)")
plt.ylabel("Seen AUROC - Hidden AUROC")
plt.show()

## 4. Summary Table

Final scores and random baseline comparison.

In [ ]:
summary = {
    "Metric": ["Seen AUROC", "Hidden AUROC", "Inductive Gap", "Seen AP", "Hidden AP", "Baseline AUROC"],
    "GraphSAGE": [
        sage['test_seen_users_edge_split']['auroc'],
        sage['inductive_hidden_user_test']['auroc'],
        sage['test_seen_users_edge_split']['auroc'] - sage['inductive_hidden_user_test']['auroc'],
        sage['test_seen_users_edge_split']['ap'],
        sage['inductive_hidden_user_test']['ap'],
        sage['inductive_baseline_random_auroc']
    ],
    "GCN": [
        gcn['seen_user_test']['value']['auroc'],
        gcn['inductive_hidden_user_test']['value']['auroc'],
        gcn['seen_user_test']['value']['auroc'] - gcn['inductive_hidden_user_test']['value']['auroc'],
        gcn['seen_user_test']['value']['ap'],
        gcn['inductive_hidden_user_test']['value']['ap'],
        gcn['inductive_baseline_random_auroc']
    ]
}

summary_df = pd.DataFrame(summary).set_index("Metric")
display(summary_df.round(4))

## Conclusion

- **GraphSAGE** typically handles the inductive setting better because it learns neighborhood aggregation functions that generalize to new nodes.
- **GCN** is powerful in transductive settings but often struggles more with the inductive gap, especially in this bipartite heterogeneous setup where we had to adapt the convolution to handle (user, reviews, item) edges.